In [1]:
"""
FiLM-HyperPINN baseline for 1D linear advection with periodic BC and Gaussian IC.

PDE:
    u_t + u_x = 0,   x in [0,1), t in [0,1]
with periodic boundary condition and task-dependent Gaussian initial condition
    u(x,0) = exp( - d_per(x, x0)^2 / (2 nu^2) )

This script is intentionally matched to the KAPI notebook TEST_CASE_02_KAPI.ipynb:
- same task parameters p = [x0, nu]
- same training ranges and curriculum in nu
- same collocation/IC/periodic sampling
- same exact solution for evaluation
- same test cases and multicase plots
"""

import os
import csv
import math
import random
import argparse
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt


# ============================================================
# Reproducibility
# ============================================================
SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ============================================================
# Utilities
# ============================================================
def get_device(use_gpu_flag: bool) -> torch.device:
    if use_gpu_flag and torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def periodic_distance_torch(x: torch.Tensor, mu: torch.Tensor) -> torch.Tensor:
    return torch.remainder(x - mu + 0.5, 1.0) - 0.5


def periodic_distance_numpy(x: np.ndarray, mu: np.ndarray) -> np.ndarray:
    return np.remainder(x - mu + 0.5, 1.0) - 0.5


def periodic_gaussian_torch(x: torch.Tensor, x0: torch.Tensor, nu: torch.Tensor) -> torch.Tensor:
    d = periodic_distance_torch(x, x0)
    return torch.exp(-(d ** 2) / (2.0 * nu ** 2))


def periodic_gaussian_numpy(x: np.ndarray, x0: float, nu: float) -> np.ndarray:
    d = periodic_distance_numpy(x, x0)
    return np.exp(-(d ** 2) / (2.0 * nu ** 2))


def fourier_features(t: torch.Tensor, n_freq: int) -> torch.Tensor:
    feats = [t]
    for k in range(1, n_freq + 1):
        feats.append(torch.sin(2.0 * math.pi * k * t))
        feats.append(torch.cos(2.0 * math.pi * k * t))
    return torch.cat(feats, dim=-1)


def periodic_x_features(x: torch.Tensor, n_freq: int) -> torch.Tensor:
    feats = [x]
    for k in range(1, n_freq + 1):
        feats.append(torch.sin(2.0 * math.pi * k * x))
        feats.append(torch.cos(2.0 * math.pi * k * x))
    return torch.cat(feats, dim=-1)


# ============================================================
# Model
# ============================================================
class FiLMHyperPINNAdvection(nn.Module):
    """
    Trunk: coordinate MLP on periodic-x and Fourier-t features.
    Hypernet: p=[x0,nu] -> FiLM parameters for each hidden layer.

    Output is unconstrained; periodic BC and IC are enforced through losses,
    matched to the KAPI training setup.
    """

    def __init__(
        self,
        trunk_width: int = 128,
        trunk_depth: int = 4,
        hyper_width: int = 64,
        n_fourier_t: int = 4,
        n_fourier_x: int = 2,
    ) -> None:
        super().__init__()
        assert trunk_depth >= 2
        self.trunk_width = trunk_width
        self.trunk_depth = trunk_depth
        self.n_fourier_t = n_fourier_t
        self.n_fourier_x = n_fourier_x

        in_dim = (1 + 2 * n_fourier_x) + (1 + 2 * n_fourier_t)
        self.trunk_in = nn.Linear(in_dim, trunk_width)
        self.trunk_h = nn.ModuleList([nn.Linear(trunk_width, trunk_width) for _ in range(trunk_depth - 1)])
        self.trunk_out = nn.Linear(trunk_width, 1)
        self.act = nn.Tanh()

        film_blocks = trunk_depth
        film_out_dim = film_blocks * 2 * trunk_width
        self.hyper = nn.Sequential(
            nn.Linear(2, hyper_width),
            nn.Tanh(),
            nn.Linear(hyper_width, hyper_width),
            nn.Tanh(),
            nn.Linear(hyper_width, film_out_dim),
        )

        self.register_buffer("p_mean", torch.tensor([0.5, 0.075], dtype=torch.float32))
        self.register_buffer("p_std", torch.tensor([0.3, 0.03], dtype=torch.float32))
        self.bias = nn.Parameter(torch.zeros(1))

    def coord_features(self, xt: torch.Tensor) -> torch.Tensor:
        x = xt[:, :, 0:1]
        t = xt[:, :, 1:2]
        xf = periodic_x_features(x.reshape(-1, 1), self.n_fourier_x).reshape(xt.shape[0], xt.shape[1], -1)
        tf = fourier_features(t.reshape(-1, 1), self.n_fourier_t).reshape(xt.shape[0], xt.shape[1], -1)
        return torch.cat([xf, tf], dim=-1)

    def forward(self, p: torch.Tensor, xt: torch.Tensor):
        """
        p : (B,2)
        xt: (B,N,2)
        returns u: (B,N)
        """
        B, N, _ = xt.shape
        p_norm = (p - self.p_mean.to(p.device)) / self.p_std.to(p.device)
        film = self.hyper(p_norm).view(B, self.trunk_depth, 2, self.trunk_width)
        gammas = film[:, :, 0, :]
        betas = film[:, :, 1, :]

        h = self.coord_features(xt).reshape(B * N, -1)
        h = self.trunk_in(h).view(B, N, self.trunk_width)
        h = gammas[:, 0, :].unsqueeze(1) * h + betas[:, 0, :].unsqueeze(1)
        h = self.act(h)

        for li, layer in enumerate(self.trunk_h, start=1):
            h = layer(h.reshape(B * N, self.trunk_width)).view(B, N, self.trunk_width)
            h = gammas[:, li, :].unsqueeze(1) * h + betas[:, li, :].unsqueeze(1)
            h = self.act(h)

        u = self.trunk_out(h.reshape(B * N, self.trunk_width)).view(B, N) + self.bias
        return u, None


# ============================================================
# Sampling
# ============================================================
def sample_p_batch(B: int, device: torch.device, x0_range=(0.2, 0.8), nu_range=(0.03, 0.12)) -> torch.Tensor:
    x0 = x0_range[0] + (x0_range[1] - x0_range[0]) * torch.rand(B, 1, device=device)
    u = torch.rand(B, 1, device=device)
    log_nu_min = math.log10(nu_range[0])
    log_nu_max = math.log10(nu_range[1])
    nu = 10.0 ** (log_nu_min + (log_nu_max - log_nu_min) * u)
    return torch.cat([x0, nu], dim=1)


def sample_interior_points_batch(B: int, N: int, device: torch.device):
    x = torch.rand(B, N, 1, device=device)
    t = torch.rand(B, N, 1, device=device)
    return torch.cat([x, t], dim=-1)


def sample_ic_points_batch(B: int, N: int, device: torch.device):
    x = torch.rand(B, N, 1, device=device)
    t = torch.zeros(B, N, 1, device=device)
    return torch.cat([x, t], dim=-1)


def sample_periodic_points_batch(B: int, N: int, device: torch.device):
    t = torch.rand(B, N, 1, device=device)
    xL = torch.zeros(B, N, 1, device=device)
    xR = torch.ones(B, N, 1, device=device)
    return torch.cat([xL, t], dim=-1), torch.cat([xR, t], dim=-1)


def sample_near_ic_points_batch(B: int, N: int, device: torch.device, t_max: float = 0.15):
    x = torch.rand(B, N, 1, device=device)
    t = t_max * torch.rand(B, N, 1, device=device)
    return torch.cat([x, t], dim=-1)


# ============================================================
# PDE residual
# ============================================================
def advection_residual_batch(model: nn.Module, p: torch.Tensor, xt_int: torch.Tensor) -> torch.Tensor:
    xt_int = xt_int.clone().detach().requires_grad_(True)
    u, _ = model(p, xt_int)
    grads = torch.autograd.grad(
        outputs=u,
        inputs=xt_int,
        grad_outputs=torch.ones_like(u),
        create_graph=True,
        retain_graph=True,
    )[0]
    u_x = grads[:, :, 0]
    u_t = grads[:, :, 1]
    return u_t + u_x


# ============================================================
# Evaluation / plotting
# ============================================================
def exact_solution(x, t, x0, nu):
    s = np.remainder(x - t, 1.0)
    return periodic_gaussian_numpy(s, x0, nu)


def plot_training_history(history, save_path):
    plt.figure(figsize=(7, 4.5))
    for key in ["loss", "loss_pde", "loss_ic", "loss_per"]:
        plt.plot(history["epoch"], history[key], lw=1.8, label=key.replace("loss_", "").upper())
    plt.yscale("log")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training history (FiLM-HyperPINN)")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=250)
    plt.close()


def evaluate_case_on_grid(model, device, x0_test, nu_test, Nx=200, Nt=200):
    x = np.linspace(0.0, 1.0, Nx, endpoint=False)
    t = np.linspace(0.0, 1.0, Nt)
    X, T = np.meshgrid(x, t, indexing="ij")

    xt_grid = np.stack([X.ravel(), T.ravel()], axis=1)
    xt_grid_t = torch.tensor(xt_grid, dtype=torch.float32, device=device).view(1, -1, 2)
    p_test = torch.tensor([[x0_test, nu_test]], dtype=torch.float32, device=device)

    model.eval()
    with torch.no_grad():
        u_pred_flat, _ = model(p_test, xt_grid_t)
    u_pred = u_pred_flat.cpu().numpy().reshape(Nx, Nt)

    u_exact = exact_solution(X, T, x0_test, nu_test)
    abs_err = np.abs(u_pred - u_exact)
    rel_l2 = np.linalg.norm((u_pred - u_exact).ravel()) / (np.linalg.norm(u_exact.ravel()) + 1e-14)
    return {
        "u_exact": u_exact,
        "u_pred": u_pred,
        "abs_err": abs_err,
        "rel_l2": rel_l2,
        "x": x,
        "t": t,
    }


def plot_multicase_fields(model, device, test_cases, save_dir, Nx=200, Nt=200, bundle_name="hyperpinn_bundle.npz"):
    num_tests = len(test_cases)
    fig, axs = plt.subplots(3, num_tests, figsize=(4.4 * num_tests, 9.4), constrained_layout=True)
    if num_tests == 1:
        axs = axs.reshape(3, 1)

    results = []
    exacts, preds, errs = [], [], []
    case_names, params = [], []
    all_u_exact, all_u_pred, all_abs_err = [], [], []
    x_vec = t_vec = None

    for name, x0, nu in test_cases:
        out = evaluate_case_on_grid(model, device, x0, nu, Nx=Nx, Nt=Nt)
        exacts.append((name, x0, nu, out["u_exact"]))
        preds.append((name, x0, nu, out["u_pred"]))
        errs.append((name, x0, nu, out["abs_err"]))
        results.append({"name": name, "x0": x0, "nu": nu, "rel_l2": out["rel_l2"]})

        case_names.append(name)
        params.append([x0, nu])
        all_u_exact.append(out["u_exact"].astype(np.float32))
        all_u_pred.append(out["u_pred"].astype(np.float32))
        all_abs_err.append(out["abs_err"].astype(np.float32))
        if x_vec is None:
            x_vec = out["x"].astype(np.float32)
            t_vec = out["t"].astype(np.float32)

    sol_vals = np.concatenate([u.ravel() for *_, u in exacts + preds])
    err_vals = np.concatenate([u.ravel() for *_, u in errs])
    sol_vmin, sol_vmax = float(sol_vals.min()), float(sol_vals.max())
    err_vmax = float(err_vals.max())
    im_sol = None
    im_err = None

    for j in range(num_tests):
        name, x0, nu, ue = exacts[j]
        _, _, _, up = preds[j]
        _, _, _, er = errs[j]

        ax = axs[0, j]
        im_sol = ax.imshow(ue.T, origin="lower", extent=[0, 1, 0, 1], aspect="auto", vmin=sol_vmin, vmax=sol_vmax)
        ax.set_title(rf"$(x_0,\nu)=({x0:.2f},{nu:.2f})$")
        ax.set_xlabel("x")
        if j == 0:
            ax.set_ylabel("Exact")

        ax = axs[1, j]
        ax.imshow(up.T, origin="lower", extent=[0, 1, 0, 1], aspect="auto", vmin=sol_vmin, vmax=sol_vmax)
        ax.set_xlabel("x")
        if j == 0:
            ax.set_ylabel("Prediction")

        ax = axs[2, j]
        im_err = ax.imshow(er.T, origin="lower", extent=[0, 1, 0, 1], aspect="auto", vmin=0.0, vmax=err_vmax)
        ax.set_xlabel("x")
        if j == 0:
            ax.set_ylabel(r"$|u_{pred}-u_{exact}|$")

    cbar1 = fig.colorbar(im_sol, ax=axs[0:2, :], shrink=0.95, location="right")
    cbar1.set_label("Solution value")
    cbar2 = fig.colorbar(im_err, ax=axs[2, :], shrink=0.95, location="right")
    cbar2.set_label("Absolute error")

    fig.savefig(os.path.join(save_dir, "multicase_fields_hyperpinn.png"), dpi=250, bbox_inches="tight")
    plt.close(fig)

    np.savez_compressed(
        os.path.join(save_dir, bundle_name),
        x=x_vec,
        t=t_vec,
        params=np.asarray(params, dtype=np.float32),
        case_names=np.asarray(case_names),
        u_exact=np.stack(all_u_exact, axis=0),
        u_pred=np.stack(all_u_pred, axis=0),
        abs_error=np.stack(all_abs_err, axis=0),
        method=np.asarray("HyperPINN"),
    )
    return results


def save_summary_csv(summary, save_dir):
    path = os.path.join(save_dir, "test_case_summary.csv")
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["name", "x0", "nu", "rel_l2"])
        writer.writeheader()
        for row in summary:
            writer.writerow(row)


# ============================================================
# Training
# ============================================================
@dataclass
class TrainConfig:
    B_pde: int = 4
    N_int: int = 256
    N_ic: int = 128
    N_per: int = 128
    N_near: int = 96
    epochs: int = 5000
    lr: float = 1e-3
    print_every: int = 100
    nu_min: float = 0.03
    nu_max: float = 0.12
    nu_easy: float = 0.06
    lambda_ic: float = 10.0
    lambda_per: float = 1.0


def train_hyperpinn(model: nn.Module, device: torch.device, cfg: TrainConfig):
    optimizer = optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=1e-6)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=max(cfg.epochs // 2, 1), gamma=0.5)
    best_loss = float("inf")
    best_state = None

    history = {k: [] for k in ["epoch", "loss", "loss_pde", "loss_ic", "loss_per"]}

    for ep in range(1, cfg.epochs + 1):
        model.train()
        optimizer.zero_grad()

        if ep <= cfg.epochs // 2:
            nu_min_curr, nu_max_curr = cfg.nu_easy, cfg.nu_max
        else:
            nu_min_curr, nu_max_curr = cfg.nu_min, cfg.nu_max

        p_batch = sample_p_batch(cfg.B_pde, device, nu_range=(nu_min_curr, nu_max_curr))
        xt_int = sample_interior_points_batch(cfg.B_pde, cfg.N_int, device)
        xt_ic = sample_ic_points_batch(cfg.B_pde, cfg.N_ic, device)
        xtL, xtR = sample_periodic_points_batch(cfg.B_pde, cfg.N_per, device)
        xt_near = sample_near_ic_points_batch(cfg.B_pde, cfg.N_near, device, t_max=0.15)

        res_int = advection_residual_batch(model, p_batch, xt_int)
        res_near = advection_residual_batch(model, p_batch, xt_near)
        loss_pde = torch.mean(res_int ** 2) + 0.5 * torch.mean(res_near ** 2)

        u_ic, _ = model(p_batch, xt_ic)
        x_ic = xt_ic[:, :, 0]
        x0 = p_batch[:, 0:1].expand_as(x_ic)
        nu = p_batch[:, 1:2].expand_as(x_ic)
        u_ic_true = periodic_gaussian_torch(x_ic, x0, nu)
        loss_ic = torch.mean((u_ic - u_ic_true) ** 2)

        uL, _ = model(p_batch, xtL)
        uR, _ = model(p_batch, xtR)
        loss_per = torch.mean((uL - uR) ** 2)

        loss = loss_pde + cfg.lambda_ic * loss_ic + cfg.lambda_per * loss_per
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        scheduler.step()

        history["epoch"].append(ep)
        history["loss"].append(loss.item())
        history["loss_pde"].append(loss_pde.item())
        history["loss_ic"].append(loss_ic.item())
        history["loss_per"].append(loss_per.item())

        if loss.item() < best_loss:
            best_loss = loss.item()
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if ep % cfg.print_every == 0 or ep == 1:
            lr_now = scheduler.get_last_lr()[0]
            print(
                f"Epoch {ep:4d} | Loss {loss.item():.3e} | PDE {loss_pde.item():.3e} | "
                f"IC {loss_ic.item():.3e} | PER {loss_per.item():.3e} | lr {lr_now:.1e}"
            )

        if device.type == "cuda":
            torch.cuda.empty_cache()

    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    return model, history


# ============================================================
# Main
# ============================================================
def main():
    parser = argparse.ArgumentParser(description="FiLM-HyperPINN for parametric linear advection")
    parser.add_argument("--gpu", action="store_true", help="Use GPU if available")
    parser.add_argument("--epochs", type=int, default=5000)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--outdir", type=str, default="outputs_hyperpinn")
    parser.add_argument("--model_path", type=str, default="hyperpinn_advection.pt")
    args, _unknown = parser.parse_known_args()

    os.makedirs(args.outdir, exist_ok=True)
    device = get_device(args.gpu)
    print("Using device:", device)

    cfg = TrainConfig(epochs=args.epochs, lr=args.lr)
    model = FiLMHyperPINNAdvection().to(device)
    model, history = train_hyperpinn(model, device, cfg)

    model_path = os.path.join(args.outdir, args.model_path)
    torch.save(model.state_dict(), model_path)
    print(f"Model saved to: {model_path}")

    plot_training_history(history, os.path.join(args.outdir, "training_loss.png"))

    loaded_model = FiLMHyperPINNAdvection().to(device)
    loaded_model.load_state_dict(torch.load(model_path, map_location=device))
    loaded_model.eval()

    test_cases = [
        ("Center_in_range", 0.50, 0.07),
        ("Offcenter_in_range", 0.30, 0.09),
        ("Narrow_in_range", 0.65, 0.04),
        ("Narrow_out_range", 0.50, 0.02),
    ]
    summary = plot_multicase_fields(loaded_model, device, test_cases, args.outdir, Nx=200, Nt=200)
    save_summary_csv(summary, args.outdir)

    print("\nTest summary:")
    for row in summary:
        print(f"{row['name']:20s} | x0={row['x0']:.3f}, nu={row['nu']:.3f} | relL2={row['rel_l2']:.3e}")
    print(f"\nAll outputs saved in: {args.outdir}")


if __name__ == "__main__":
    main()


Using device: cpu
Epoch    1 | Loss 1.539e+00 | PDE 5.414e-07 | IC 1.539e-01 | PER 3.921e-11 | lr 1.0e-03
Epoch  100 | Loss 8.849e-01 | PDE 4.898e-02 | IC 8.355e-02 | PER 4.363e-04 | lr 1.0e-03
Epoch  200 | Loss 4.563e-01 | PDE 1.869e-01 | IC 2.568e-02 | PER 1.253e-02 | lr 1.0e-03
Epoch  300 | Loss 1.550e-01 | PDE 8.988e-02 | IC 4.698e-03 | PER 1.814e-02 | lr 1.0e-03
Epoch  400 | Loss 1.067e-01 | PDE 4.661e-02 | IC 2.293e-03 | PER 3.719e-02 | lr 1.0e-03
Epoch  500 | Loss 8.908e-02 | PDE 3.938e-02 | IC 1.305e-03 | PER 3.665e-02 | lr 1.0e-03
Epoch  600 | Loss 7.856e-02 | PDE 3.907e-02 | IC 1.855e-03 | PER 2.094e-02 | lr 1.0e-03
Epoch  700 | Loss 8.173e-02 | PDE 4.828e-02 | IC 1.129e-03 | PER 2.216e-02 | lr 1.0e-03
Epoch  800 | Loss 5.498e-02 | PDE 2.751e-02 | IC 5.708e-04 | PER 2.176e-02 | lr 1.0e-03
Epoch  900 | Loss 6.004e-02 | PDE 3.751e-02 | IC 8.912e-04 | PER 1.362e-02 | lr 1.0e-03
Epoch 1000 | Loss 5.297e-02 | PDE 2.628e-02 | IC 1.514e-03 | PER 1.155e-02 | lr 1.0e-03
Epoch 1100 | L

/tmp/ipykernel_28686/1857813428.py:479: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  loaded_model.load_state_dict(torch.load(model_path, map_location=device))



Test summary:
Center_in_range      | x0=0.500, nu=0.070 | relL2=5.581e-02
Offcenter_in_range   | x0=0.300, nu=0.090 | relL2=4.203e-02
Narrow_in_range      | x0=0.650, nu=0.040 | relL2=2.047e-01
Narrow_out_range     | x0=0.500, nu=0.020 | relL2=3.752e-01

All outputs saved in: outputs_hyperpinn
